# BiGRU + Attention + Metadata

This notebook trains and evaluates the BiGRU+Attention model with metadata features (genre, playtime tier, review length tier).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

import fasttext
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

with open("results/bigru_metadata_optuna_best_params.json", "r") as f:
    best_params = json.load(f)

# Configuration
RANDOM_SEED = 42
N_FOLDS = 5
BATCH_SIZE = 32
MAX_LENGTH = 256
EMBEDDING_DIM = 200
HIDDEN_DIM = best_params['best_params']['hidden_dim']
NUM_LAYERS = best_params['best_params']['num_layers']
DROPOUT = best_params['best_params']['dropout']
LEARNING_RATE = best_params['best_params']['learning_rate']
WEIGHT_DECAY = best_params['best_params']['weight_decay']
NUM_EPOCHS = 15
PATIENCE = 3
METADATA_EMBED_DIM = best_params['best_params']['metadata_embed_dim']

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Data and Encode Metadata

In [ ]:
# Load train and test data
DATA_DIR = Path('/kaggle/input/steam-review')
MODEL_DIR = Path('/kaggle/input/fasttext-steam-200/other/default/1')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Train set: {len(train_df):,} reviews")
print(f"Test set: {len(test_df):,} reviews")
print("\nTrain sentiment distribution:")
print(train_df['voted_up'].value_counts(normalize=True))

In [ ]:
# Encode categorical metadata
genre_encoder = LabelEncoder()
playtime_tier_encoder = LabelEncoder()
length_tier_encoder = LabelEncoder()

genre_encoder.fit(train_df['primary_genre'])
playtime_tier_encoder.fit(train_df['playtime_tier'])
length_tier_encoder.fit(train_df['length_tier'])

# Transform
train_df['genre_id'] = genre_encoder.transform(train_df['primary_genre'])
train_df['playtime_tier_id'] = playtime_tier_encoder.transform(train_df['playtime_tier'])
train_df['length_tier_id'] = length_tier_encoder.transform(train_df['length_tier'])

test_df['genre_id'] = genre_encoder.transform(test_df['primary_genre'])
test_df['playtime_tier_id'] = playtime_tier_encoder.transform(test_df['playtime_tier'])
test_df['length_tier_id'] = length_tier_encoder.transform(test_df['length_tier'])

NUM_GENRES = len(genre_encoder.classes_)
NUM_PLAYTIME_TIERS = len(playtime_tier_encoder.classes_)
NUM_LENGTH_TIERS = len(length_tier_encoder.classes_)

print(f"Genre categories ({NUM_GENRES}): {list(genre_encoder.classes_)}")
print(f"Playtime tier categories ({NUM_PLAYTIME_TIERS}): {list(playtime_tier_encoder.classes_)}")
print(f"Length tier categories ({NUM_LENGTH_TIERS}): {list(length_tier_encoder.classes_)}")

In [ ]:
# Prepare features and labels
X_train = train_df['processed_text'].fillna('').values
y_train = train_df['voted_up'].astype(int).values

genre_ids_train = train_df['genre_id'].values
playtime_tier_ids_train = train_df['playtime_tier_id'].values
length_tier_ids_train = train_df['length_tier_id'].values

X_test = test_df['processed_text'].fillna('').values
y_test = test_df['voted_up'].astype(int).values

genre_ids_test = test_df['genre_id'].values
playtime_tier_ids_test = test_df['playtime_tier_id'].values
length_tier_ids_test = test_df['length_tier_id'].values

stratify_key = train_df['voted_up'].astype(str) + '_' + train_df['primary_genre'].astype(str)

print(f"Features: {len(X_train):,} train, {len(X_test):,} test")

## 2. Load FastText Embeddings

In [ ]:
print("Loading FastText model...")
ft_model = fasttext.load_model(str(MODEL_DIR / 'steam_fasttext.bin'))
print(f"FastText loaded: {ft_model.get_dimension()}-dimensional embeddings")
print(f"Vocabulary size: {len(ft_model.words):,}")

## 3. Dataset with Metadata

In [ ]:
class SteamReviewDatasetWithMetadata(Dataset):
    def __init__(self, texts, labels, genre_ids, playtime_tier_ids, length_tier_ids,
                 ft_model, max_length=256):
        self.texts = texts
        self.labels = labels
        self.genre_ids = genre_ids
        self.playtime_tier_ids = playtime_tier_ids
        self.length_tier_ids = length_tier_ids
        self.ft_model = ft_model
        self.max_length = max_length
        self.embedding_dim = ft_model.get_dimension()
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = text.split()[:self.max_length]
        
        embeddings = np.zeros((self.max_length, self.embedding_dim), dtype=np.float32)
        for i, token in enumerate(tokens):
            embeddings[i] = self.ft_model.get_word_vector(token)
        
        mask = np.zeros(self.max_length, dtype=np.float32)
        mask[:len(tokens)] = 1.0
        
        return {
            'embeddings': torch.tensor(embeddings, dtype=torch.float32),
            'mask': torch.tensor(mask, dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32),
            'genre_id': torch.tensor(self.genre_ids[idx], dtype=torch.long),
            'playtime_tier_id': torch.tensor(self.playtime_tier_ids[idx], dtype=torch.long),
            'length_tier_id': torch.tensor(self.length_tier_ids[idx], dtype=torch.long)
        }


def create_dataloader_with_metadata(texts, labels, genre_ids, playtime_tier_ids, 
                                     length_tier_ids, ft_model, batch_size, shuffle=True):
    dataset = SteamReviewDatasetWithMetadata(
        texts, labels, genre_ids, playtime_tier_ids, length_tier_ids,
        ft_model, MAX_LENGTH
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=True
    )

## 4. Model Architecture

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False)
        )
    
    def forward(self, gru_output, mask):
        scores = self.attention(gru_output).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e4)
        attention_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attention_weights.unsqueeze(1), gru_output).squeeze(1)
        return context, attention_weights


class BiGRUAttentionWithMetadata(nn.Module):
    """FastText + BiGRU + Attention + Metadata + FNN for sentiment classification."""
    
    def __init__(self, embedding_dim=200, hidden_dim=128, num_layers=2, dropout=0.4,
                 num_genres=12, num_playtime_tiers=4, num_length_tiers=3, 
                 metadata_embed_dim=16):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.attention = Attention(hidden_dim * 2)
        
        # Metadata embeddings
        self.genre_embed = nn.Embedding(num_genres, metadata_embed_dim)
        self.playtime_tier_embed = nn.Embedding(num_playtime_tiers, metadata_embed_dim)
        self.length_tier_embed = nn.Embedding(num_length_tiers, metadata_embed_dim)
        
        combined_dim = hidden_dim * 2 + metadata_embed_dim * 3
        
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, embeddings, mask, genre_ids, playtime_tier_ids, length_tier_ids):
        self.gru.flatten_parameters()
        gru_output, _ = self.gru(embeddings)
        text_context, _ = self.attention(gru_output, mask)
        
        genre_emb = self.genre_embed(genre_ids)
        playtime_emb = self.playtime_tier_embed(playtime_tier_ids)
        length_emb = self.length_tier_embed(length_tier_ids)
        
        combined = torch.cat([text_context, genre_emb, playtime_emb, length_emb], dim=-1)
        return self.classifier(combined)

In [ ]:
# Instantiate model
model = BiGRUAttentionWithMetadata(
    embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    num_genres=NUM_GENRES, num_playtime_tiers=NUM_PLAYTIME_TIERS, 
    num_length_tiers=NUM_LENGTH_TIERS, metadata_embed_dim=METADATA_EMBED_DIM
)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training Functions

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for batch in tqdm(dataloader, desc='Training', leave=False):
        embeddings = batch['embeddings'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['label'].to(device)
        genre_ids = batch['genre_id'].to(device)
        playtime_tier_ids = batch['playtime_tier_id'].to(device)
        length_tier_ids = batch['length_tier_id'].to(device)
        
        optimizer.zero_grad()
        
        with autocast(device_type=device.type):
            logits = model(embeddings, mask, genre_ids, playtime_tier_ids, length_tier_ids).squeeze(-1)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            embeddings = batch['embeddings'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            genre_ids = batch['genre_id'].to(device)
            playtime_tier_ids = batch['playtime_tier_id'].to(device)
            length_tier_ids = batch['length_tier_id'].to(device)
            
            with autocast(device_type=device.type):
                logits = model(embeddings, mask, genre_ids, playtime_tier_ids, length_tier_ids).squeeze(-1)
                loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    return avg_loss, accuracy, precision, recall, f1, all_preds, all_labels

## 6. K-Fold Cross-Validation

In [ ]:
print(f"Running {N_FOLDS}-Fold Cross-Validation")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_results = []
best_epochs_per_fold = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, stratify_key)):
    print(f"\nFold {fold + 1}/{N_FOLDS}")
    
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    genre_fold_train, genre_fold_val = genre_ids_train[train_idx], genre_ids_train[val_idx]
    playtime_fold_train, playtime_fold_val = playtime_tier_ids_train[train_idx], playtime_tier_ids_train[val_idx]
    length_fold_train, length_fold_val = length_tier_ids_train[train_idx], length_tier_ids_train[val_idx]
    
    train_loader = create_dataloader_with_metadata(
        X_fold_train, y_fold_train, genre_fold_train, playtime_fold_train, length_fold_train,
        ft_model, BATCH_SIZE, shuffle=True
    )
    val_loader = create_dataloader_with_metadata(
        X_fold_val, y_fold_val, genre_fold_val, playtime_fold_val, length_fold_val,
        ft_model, BATCH_SIZE, shuffle=False
    )
    
    model = BiGRUAttentionWithMetadata(
        embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
        num_genres=NUM_GENRES, num_playtime_tiers=NUM_PLAYTIME_TIERS,
        num_length_tiers=NUM_LENGTH_TIERS, metadata_embed_dim=METADATA_EMBED_DIM
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1, verbose=True)
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler()
    
    best_val_f1 = 0
    best_epoch = 0
    patience_counter = 0
    best_metrics = None
    
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
        val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        
        scheduler.step(val_f1)
        print(f"Epoch {epoch + 1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch + 1
            best_metrics = {'accuracy': val_acc, 'precision': val_prec, 'recall': val_rec, 'f1': val_f1}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch + 1}")
                break
    
    fold_results.append(best_metrics)
    best_epochs_per_fold.append(best_epoch)
    print(f"Fold {fold + 1} Best: Accuracy={best_metrics['accuracy']:.4f}, F1={best_metrics['f1']:.4f}")

In [ ]:
# CV Summary
print("\n" + "=" * 60)
print("BiGRU+Attention+Metadata Cross-Validation Results")
print("=" * 60)

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    values = [r[metric] for r in fold_results]
    print(f"{metric.capitalize()}: {np.mean(values):.4f} ± {np.std(values):.4f}")

print(f"\nBest epochs per fold: {best_epochs_per_fold}")
optimal_epochs = int(np.median(best_epochs_per_fold))
print(f"Optimal epochs for final training: {optimal_epochs}")

## 7. Final Training and Test Evaluation

In [ ]:
print("\nTraining final model on full training data...")

full_train_loader = create_dataloader_with_metadata(
    X_train, y_train, genre_ids_train, playtime_tier_ids_train, length_tier_ids_train,
    ft_model, BATCH_SIZE, shuffle=True
)

test_loader = create_dataloader_with_metadata(
    X_test, y_test, genre_ids_test, playtime_tier_ids_test, length_tier_ids_test,
    ft_model, BATCH_SIZE, shuffle=False
)

final_model = BiGRUAttentionWithMetadata(
    embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    num_genres=NUM_GENRES, num_playtime_tiers=NUM_PLAYTIME_TIERS,
    num_length_tiers=NUM_LENGTH_TIERS, metadata_embed_dim=METADATA_EMBED_DIM
).to(device)

optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()

for epoch in range(optimal_epochs):
    train_loss, train_acc = train_epoch(final_model, full_train_loader, optimizer, criterion, scaler, device)
    print(f"Epoch {epoch + 1}/{optimal_epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}")

In [ ]:
# Test evaluation
print("\nEvaluating on test set...")

test_loss, test_acc, test_prec, test_rec, test_f1, test_preds, test_labels = evaluate(
    final_model, test_loader, criterion, device
)

print("\n" + "=" * 60)
print("TEST SET RESULTS - BiGRU+Attention+Metadata")
print("=" * 60)
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_rec:.4f}")
print(f"F1 Score:  {test_f1:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - BiGRU+Attention+Metadata')
plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
import json

results = {
    'model_type': 'BiGRU+Attention+Metadata',
    'cv_results': {
        'accuracy': {'mean': float(np.mean([r['accuracy'] for r in fold_results])),
                     'std': float(np.std([r['accuracy'] for r in fold_results]))},
        'precision': {'mean': float(np.mean([r['precision'] for r in fold_results])),
                      'std': float(np.std([r['precision'] for r in fold_results]))},
        'recall': {'mean': float(np.mean([r['recall'] for r in fold_results])),
                   'std': float(np.std([r['recall'] for r in fold_results]))},
        'f1': {'mean': float(np.mean([r['f1'] for r in fold_results])),
               'std': float(np.std([r['f1'] for r in fold_results]))}
    },
    'test_results': {
        'accuracy': float(test_acc),
        'precision': float(test_prec),
        'recall': float(test_rec),
        'f1': float(test_f1)
    },
    'optimal_epochs': optimal_epochs,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'learning_rate': LEARNING_RATE,
        'metadata_embed_dim': METADATA_EMBED_DIM
    }
}

print(json.dumps(results, indent=2))